<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_00_transient_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 · Notebook 00 — Transient Setup

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L8.2 · Dynamic Heat**

## What Ex_08.2 is about

Transient conduction, solved by a physics-informed network on a space–time
slab. The set compares soft and hard initial conditions on a square plate with
an exact solution, puts the same equation in time on a plate with an elliptical
hole, where there is no exact solution to score against, and ends with an
inverse problem: recovering an unknown **diffusivity** from a noisy cooling
curve. That last step asks the network for a physical constant rather than a
field, and raises the question of whether the data supports the number it
returns.

### Goals

By the end of the exercise set you can

1. write a first-order-in-time residual on a space–time slab and say which
   column of the gradient is the time derivative;
2. assemble a three-term loss and report **which** term the optimiser is
   actually reducing, rather than only the total;
3. enforce an initial condition and a boundary condition exactly, by
   construction, and state precisely what that costs the network;
4. put the same equation on a domain with a hole, using a level set as both the
   rejection test and the hard-boundary multiplier;
5. recover an unknown diffusivity from noisy data, and decide whether the data
   supports the number you got — **identifiability**;
6. choose a time window from the diffusion time constant, and report relative
   *and* absolute error because at late times one of them is meaningless.

### Method — six notebooks, run in order

| notebook | what you do | problem | lecture |
|---|---|---|---|
| **00** | nothing to write — timescales, space–time sampling, the geometry | all three | — |
| **01** | three soft loss terms — and where the error goes | the square plate | L8.2 |
| **02** | a hard initial condition, $(1-t)\,f_{IC} + t\,D\,N$, and what $N$ pays for it | the square plate | L8.2 |
| **03** | the plate with a hole, in time | the plate with a hole | L8.2 |
| **04** | recovering the diffusivity from noisy sensor readings | the square plate, diffusivity unknown | L8.2 |
| **05** | the report | the saved results | — |

Notebooks 01 to 04 have `# TODO` cells to complete; notebooks 02 and 05 load
results the earlier ones saved in `Ex08.2_outputs/`. Everything runs on a CPU;
budget three to five minutes for each training run in 01, 02 and 03, and about
an hour for the full set.

### Applications

* **Cooling transients.** A slab held at zero on both faces has time constant
  $\tau = L^2/(\pi^2 c) = 0.1013$ and is essentially settled by $4\tau$; the
  square decays twice as fast, $\tau = L^2/(2\pi^2 c) = 0.0507$. Choosing the
  time window from $\tau$ is the first design decision in any transient model.
* **Heated parts with cooling features.** A plate with an elliptical hole held
  at zero and a uniform source switched on at $t = 0$ — the normal situation in
  practice, with no exact solution to check against.
* **Material identification.** Five sensors, twelve noisy readings each through
  the early transient, give the diffusivity as a trainable parameter; move the
  sensors into the settled part of the curve and a confident, wrong answer
  comes back.

## What this notebook checks

**Read and run; you are not asked to rewrite this.**

$$\frac{\partial T}{\partial t} = \alpha\left(
\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2}\right)$$

on the unit square, zero on all edges, with $T(x,y,0)=\sin(\pi x)\sin(\pi y)$.
Exact solution — the fundamental mode of the 2-D heat equation on the unit square with zero edges, by separation of variables:

$$T = e^{-2\pi^2 c t}\,\sin(\pi x)\sin(\pi y)$$

`problem.py` provides the plate and its elliptical hole (level set, hard-boundary
multiplier, normal), the space–time sampler that can cut the hole out of the
slab, the exact cooling solution with its time constant, the flux balance, and
the helpers that slice a model at one instant and score its error against time.
The network, the derivatives, the samplers for the initial slice and the edges
in time, the optimiser and the error metrics come from `pinn_core.py`, the
library shared by every Part 2 exercise set.

Run this first, top to bottom. Setup instructions are in `README.md`.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · Timescales — choose `t_end` before you sample

A transient problem has a clock, and the first decision in the exercise is how
much of it to solve. Solve for far longer than the diffusion time and you have
computed a steady state, correctly, at considerable expense.

In [ ]:
C = 1.0
pb.describe_problem(c=C, t_end=1.0)

**What you should see.**

```
    time constant   : tau = 0.1013   steady by about 4 tau = 0.405
    the square      : decays twice as fast, tau = 0.0507, so 4 tau is conservative

    t        Fo       amplitude
    0.0      0.000    1.000e+00
    0.05     0.050    3.727e-01
    0.1      0.100    1.389e-01
    0.5      0.500    5.172e-05
    1.0      1.000    2.675e-09
```

The amplitude falls to $e^{-1}$ at $t = 0.0507$, not at $0.1013$: $\tau = L^2/(\pi^2 c)$
is the slab's time constant, and the square's lowest mode carries $\pi^2$ once
for each dimension. Note the last column: by $t = 1$ the amplitude is ~3e-9. Late-time *relative*
errors compare two numbers that are both nearly zero — read absolute error
there. `pb.error_vs_time` returns both for exactly this reason.

---

## 2 · The space–time sample

Three sets of points, three different jobs. The samplers return **NumPy
arrays**, not tensors: they can then be plotted, saved and checked without a
device or a graph. Call `to_tensor(...)` at the point of use, with
`requires_grad=True` for anything you will differentiate through.

In [ ]:
T_END = 1.0
xyt_f = pb.plate_spacetime_points(3000, t_end=T_END)
xyt_b = boundary_points_in_time(25, 20, pb.PLATE_DOMAIN, (0.0, T_END), seed=1)
xyt_0 = initial_points(400, pb.PLATE_DOMAIN, t0=0.0, seed=1)

for nm, s in [("interior", xyt_f), ("boundary", xyt_b), ("initial", xyt_0)]:
    print(f"{nm:>9s}: {s.shape[0]:5d}   array {type(s).__name__}")
print("initial slice all at t = 0:", bool((xyt_0[:, 2] == 0).all()))

fig = plt.figure(figsize=(6.5, 5)); ax = fig.add_subplot(111, projection="3d")
for s, lab, sz in [(xyt_f, "interior", 2), (xyt_b, "boundary", 3),
                   (xyt_0, "initial", 5)]:
    ax.scatter(s[:, 0], s[:, 1], s[:, 2], s=sz, label=lab, alpha=0.6)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("t"); ax.legend(); plt.show()

The initial points form one flat face — the measure-zero slice L8.2 draws when
it makes time a coordinate.
However many points you place there, they are outnumbered by the interior, and
the optimiser is free to trade accuracy on that face for a smaller residual in
the bulk. Notebook 01 measures the trade; notebook 02 removes it.

---

## 3 · Verify the reference solution with autograd

Time is the **last** column, everywhere in this course. For a point sampled as
$(x, y, t)$ the time derivative is therefore `grad(u, xyt)[:, 2:3]`. Getting
the column wrong is a silent error — the code runs and solves a different
equation — so it is checked once, here.

In [ ]:
xyt = to_tensor(pb.plate_spacetime_points(400, t_end=T_END), requires_grad=True)
x, y, t = xyt[:, 0:1], xyt[:, 1:2], xyt[:, 2:3]

u = pb.exact_transient(x, y, t, C)
res = grad(u, xyt)[:, 2:3] - C * (d2(u, xyt, 0) + d2(u, xyt, 1))

print(f"max |residual| = {res.abs().max().item():.3e}")
check("reference solution satisfies the PDE", res.abs().max().item(), 0.0,
      tol=1e-8)

**What you should see.** One `PASS`, with the residual far below the 1e-8
tolerance — this is float64 round-off in a formula that is exact, not an
approximation error.

`pb.exact_transient` dispatches on the type of its first argument, so the same
formula served the tensors above and will serve the NumPy grids below.

---

## 4 · The other domain: the plate with a hole

Notebook 03 puts the same equation on a plate with an elliptical hole. The
hole is a level set — negative inside, zero on the boundary, positive in the
material — and that single function does two jobs: it rejects collocation points that fall in
the hole, and it *is* the multiplier that hard-enforces $T = 0$ on the hole
without any trained network.

In [ ]:
xy_plate = pb.plate_points(1500, seed=1)
xy_hole = pb.hole_points(120)

print(f"plate points : {xy_plate.shape}   min level set "
      f"{pb.ellipse_phi(xy_plate).min():.4f}   (positive: none in the hole)")
print(f"hole points  : {xy_hole.shape}   max |level set| "
      f"{np.abs(pb.ellipse_phi(xy_hole)).max():.2e}   (zero: on the boundary)")

fig, ax = plt.subplots(figsize=(5.2, 5.0))
ax.scatter(xy_plate[:, 0], xy_plate[:, 1], s=3, alpha=0.5, color="#1f77b4",
           label="interior")
ax.scatter(xy_hole[:, 0], xy_hole[:, 1], s=9, color="#d94f2b",
           label="hole, by arc length")
ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("The plate with a hole"); ax.legend(fontsize=8, frameon=False)
plt.show()

**What you should see.** A square of blue points with a clean elliptical gap,
and an orange outline whose points crowd slightly at the two sharp ends. That
crowding is deliberate: the hole is sampled by **arc length**, not by angle,
because uniform angular spacing under-samples the high-curvature ends — which
is exactly where the flux concentrates.

---

## 5 · Ready

Next: **[notebook 01](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_01_soft_ic.ipynb)**, where the initial condition is a penalty in the loss and
the plate is solved for the first time.